In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import shapes
from shapely.geometry import shape
import folium
from folium import FeatureGroup, GeoJson
import json

In [ ]:
# Categorias de vegetação/cobertura — valores MapBiomas dentro da mancha urbana
# 0 = fora da máscara | 240/241 = área construída/urbanizada (excluídos)
CATEGORIAS = {
    3:  ("Formação Florestal",       "#1f8d49"),
    4:  ("Formação Savânica",        "#7dc975"),
    5:  ("Mangue",                   "#04381d"),
    9:  ("Silvicultura",             "#7a5900"),
    11: ("Campo Alagado / Várzea",   "#519799"),
    12: ("Formação Campestre",       "#d6bc74"),
    15: ("Pastagem",                 "#edde8e"),
    20: ("Cana-de-açúcar",           "#db4035"),
    21: ("Mosaico de Usos",          "#ffefc3"),
    23: ("Praia / Duna / Areal",     "#ff8c00"),
    25: ("Outras Áreas não Veg.",    "#d4271e"),
    29: ("Afloramento Rochoso",      "#9c9c9c"),
    30: ("Mineração",                "#9e9d9d"),
    33: ("Rios e Lagos",             "#2532e4"),
    39: ("Soja",                     "#f5b3be"),
    41: ("Outras Lavouras Temp.",    "#c71585"),
    46: ("Café",                     "#d082de"),
    47: ("Citrus",                   "#e6ccff"),
    48: ("Outras Lavouras Perenes",  "#982c9e"),
    49: ("Lavoura Perene",           "#e787f8"),
    50: ("Restinga Arborizada",      "#347812"),
    75: ("Cerrado Rupestre",         "#b8af4f"),
}

TIF_DIR  = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Vegetacao_mapbiomas"
LIMITE   = "./data/green_jundiai/limite_municipal_jundiai.shp"
OUTDIR   = "./data/veg_urbana"

import os
os.makedirs(OUTDIR, exist_ok=True)

In [ ]:
# Limite de Jundiaí reprojetado para o CRS do raster (EPSG:4326)
limite = gpd.read_file(LIMITE).to_crs(4326)
limite_geom = [limite.union_all().__geo_interface__]

In [ ]:
# Lista todos os valores únicos presentes no TIF dentro do limite de Jundiaí
ANO_INSPECAO = 2022
tif_path = f"{TIF_DIR}/mapbiomas_urbano_sp_{ANO_INSPECAO}.tif"

with rasterio.open(tif_path) as src:
    lim = limite.to_crs(src.crs)
    arr_raw, _ = rio_mask(src, [lim.union_all().__geo_interface__], crop=True, filled=True, nodata=0)
    arr_raw = arr_raw[0].astype(np.uint16)

valores_unicos = sorted(int(v) for v in np.unique(arr_raw) if v != 0)

print(f"Valores encontrados no TIF {ANO_INSPECAO} ({len(valores_unicos)} categorias):\n")
print(f"{'Valor':>6}  {'Categoria':<35}  {'Mapeado?'}")
print("-" * 58)
for v in valores_unicos:
    nome, _ = CATEGORIAS.get(v, ("— não mapeado —", None))
    mapeado = "✓" if v in CATEGORIAS else "✗"
    print(f"{v:>6}  {nome:<35}  {mapeado}")

In [ ]:
def extrair_vegetacao(ano: int) -> gpd.GeoDataFrame:
    """Recorta o TIF ao limite de Jundiaí e vetoriza cada categoria de cobertura."""
    tif_path = f"{TIF_DIR}/mapbiomas_urbano_sp_{ano}.tif"

    with rasterio.open(tif_path) as src:
        # Reprojetar limite para CRS do raster se necessário
        lim = limite.to_crs(src.crs)
        geoms = [lim.union_all().__geo_interface__]

        arr, transform = rio_mask(src, geoms, crop=True, filled=True, nodata=0)
        arr = arr[0].astype(np.uint16)
        crs = src.crs

    features = []
    for val, (nome, cor) in CATEGORIAS.items():
        mask_bin = (arr == val).astype(np.uint8)
        if mask_bin.sum() == 0:
            continue
        for geom, _ in shapes(mask_bin, mask=mask_bin, transform=transform):
            features.append({
                "geometry": shape(geom),
                "valor": val,
                "categoria": nome,
                "cor": cor,
                "ano": ano,
            })

    if not features:
        return gpd.GeoDataFrame(columns=["geometry","valor","categoria","cor","ano"], crs=crs)

    gdf = gpd.GeoDataFrame(features, crs=crs).to_crs(4326)
    # Dissolver por categoria para reduzir número de polígonos
    gdf = gdf.dissolve(by=["valor","categoria","cor","ano"]).reset_index()
    gdf = gdf.explode(index_parts=False, ignore_index=True)
    return gdf

In [ ]:
# Processar todos os anos disponíveis
anos = [2000, 2010, 2020, 2022]
gdfs = {}

for ano in anos:
    print(f"Processando {ano}...")
    gdf = extrair_vegetacao(ano)
    gdfs[ano] = gdf
    out = f"{OUTDIR}/veg_urbana_jundiai_{ano}.gpkg"
    gdf.to_file(out, driver="GPKG")
    print(f"  {len(gdf)} polígonos → {out}")

print("Concluído.")

In [ ]:
# Processar 2023 separadamente (TIF disponível, fora da lista original)
print("Processando 2023...")
gdf_2023 = extrair_vegetacao(2023)
gdfs[2023] = gdf_2023
out_2023 = f"{OUTDIR}/veg_urbana_jundiai_2023.gpkg"
gdf_2023.to_file(out_2023, driver="GPKG")
print(f"  {len(gdf_2023)} polígonos → {out_2023}")

In [ ]:
# Resumo de área por categoria e ano
CRS_M = 31983  # SIRGAS UTM 23S (metros)

rows = []
for ano, gdf in gdfs.items():
    if gdf.empty:
        continue
    gdf_m = gdf.to_crs(CRS_M)
    gdf_m["area_ha"] = gdf_m.geometry.area / 10_000
    for cat, grp in gdf_m.groupby("categoria"):
        rows.append({"ano": ano, "categoria": cat, "area_ha": grp["area_ha"].sum()})

resumo = pd.DataFrame(rows).pivot(index="categoria", columns="ano", values="area_ha").fillna(0).round(1)
resumo

In [ ]:
# Mapa Folium — cobertura vegetal 2023 + mancha urbana 2023
ANO_MAPA = 2023
gdf_mapa = gdfs[ANO_MAPA]

# Mancha urbana do shapefile (class=1 / soil_use='urbano')
mancha_urbana = gpd.read_file("./data/soil_use_2023.shp").to_crs(4326)
mancha_urbana = mancha_urbana[mancha_urbana["soil_use"] == "urbano"]

centro = limite.geometry.union_all().centroid
m = folium.Map(location=[centro.y, centro.x], zoom_start=12, tiles=None)

folium.TileLayer("CartoDB positron", name="Carto Positron", overlay=False, control=True, show=True).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satélite (Esri)", overlay=False, control=True, show=True
).add_to(m)

# Limite municipal
folium.GeoJson(
    json.loads(limite.to_json()),
    name="Limite Jundiaí",
    control=False,
    style_function=lambda f: {"color": "#FFD700", "weight": 2, "fillOpacity": 0}
).add_to(m)

# Mancha urbana 2023
fg_urbana = FeatureGroup(name="Mancha Urbana 2023", show=True)
GeoJson(
    json.loads(mancha_urbana.to_json()),
    name=None,
    style_function=lambda f: {
        "fillColor": "#e8735a", "color": "#c0392b", "weight": 1.5, "fillOpacity": 0.3
    },
    tooltip=folium.GeoJsonTooltip(fields=["soil_use", "year"], aliases=["Uso:", "Ano:"])
).add_to(fg_urbana)
fg_urbana.add_to(m)

# Uma FeatureGroup por categoria de vegetação
for cat in gdf_mapa["categoria"].unique():
    sub = gdf_mapa[gdf_mapa["categoria"] == cat]
    cor = sub["cor"].iloc[0]
    fg = FeatureGroup(name=cat, show=True)
    GeoJson(
        json.loads(sub.to_json()),
        name=None,
        style_function=lambda f, c=cor: {
            "fillColor": c, "color": c, "weight": 0.5, "fillOpacity": 0.7
        },
        tooltip=folium.GeoJsonTooltip(fields=["categoria", "ano"], aliases=["Categoria:", "Ano:"])
    ).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
m.save(f"mapa_veg_urbana_jundiai_{ANO_MAPA}.html")
print(f"Salvo: mapa_veg_urbana_jundiai_{ANO_MAPA}.html")